<p align="left">
  <a href="https://colab.research.google.com/github/wgomezf/CNN_workshop/blob/main/Notebooks/convolayer.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" width="200">
  </a>
</p>

# Convolution$\rightarrow$ReLU$\rightarrow$Max-pooling
It illustrates the operation of a convolutional layer with one kernel, a ReLU activation function, and a max-pooling layer.

In [ ]:
#@title Load necessary libraries
import numpy as np                                                    # Numerical array operations
import matplotlib.pyplot as plt                                       # Data plotting/visualization
import tensorflow as tf                                               # Machine learning
import os                                                             # Interaction with the operating system
import cv2                                                            # Computer vision
import urllib.request                                                 # Open and read URLs

In [ ]:
#@title Function to read an image from a URL
def get_img_array(img_path, size=(224, 224)):
  # Fetch image from URL
  resp = urllib.request.urlopen(img_path)
  image_data = np.asarray(bytearray(resp.read()), dtype="uint8")
  img = cv2.imdecode(image_data, cv2.IMREAD_COLOR)

  # Convert Color to Gray
  img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

  # Resize to target size
  img = cv2.resize(img, size)

  return img

A 2D convolution is an operation where a kernel or filter slides over an input image, performing element-wise multiplication and summing the results. Is is defined as

$\left( w \ast f \right)\left( x,y \right) = \sum\limits_{i =  - a}^a {\sum\limits_{j =  - b}^b {w\left( {i,j} \right)f\left( {x - i,y - j} \right)} },$

where $f$ is a matrix of size $M \times N$ and $w$ is a filter of size $m \times n$ with $a=(m-1)/2$ and $b=(n-1)/2$. The value $w(i,j)$ is the filter coefficient.

In [ ]:
#@title 2D convolution implementation
def convolution_2d(image, kernel):

  # Convert to float
  image = np.array(image).astype(np.float32)
  kernel = np.array(kernel).astype(np.float32)

  # Flip the kernel vertically and horizontally (mathematical definition requirement)
  kernel = np.flipud(np.fliplr(kernel))

  img_h, img_w = image.shape
  ker_h, ker_w = kernel.shape

  # Calculate zero-padding margins to keep the output size identical
  pad_h = ker_h // 2
  pad_w = ker_w // 2

  # Create padded input array filled with zeros
  padded_img = np.pad(image, ((pad_h, pad_h), (pad_w, pad_w)), mode='constant')
  output = np.zeros_like(image)

  # Slide the kernel over every internal pixel
  for i in range(img_h):
      for j in range(img_w):
          # Extract the region of interest
          roi = padded_img[i : i + ker_h, j : j + ker_w]
          # Element-wise multiply and sum up the results
          output[i, j] = np.sum(roi * kernel)

  return output

Max-pooling downsamples a feature map by dividing it into non-overlapping patches, defined by a pool size ($P$), and outputs the maximum value for each patch, thereby reducing dimensionality while preserving prominent features. The stride ($S$) is the number of pixels the window shifts horizontally and vertically after each computation. The output size of the pooled feature map is calculated as:

$O_{size} = \lfloor \frac{W - P}{S} \rfloor + 1,$

where $W$ is the input dimension.

In [ ]:
#@title Max-pooling implementation
def max_pooling_2d(image):

    # Convert to float
    image = np.array(image).astype(np.float32)

    # Downsampling the image to half
    pool_size = 2
    stride = 2

    # Get dimensions of the input grayscale image
    h_in, w_in = image.shape

    # Calculate the dimensions of the output image
    h_out = (h_in - pool_size) // stride + 1
    w_out = (w_in - pool_size) // stride + 1

    # Initialize the output array with zeros
    output = np.zeros((h_out, w_out))

    # Slide the pooling window across the image
    for i in range(h_out):
        for j in range(w_out):
            # Define the current region boundaries
            start_r = i * stride
            end_r = start_r + pool_size
            start_c = j * stride
            end_c = start_c + pool_size

            # Extract the regional patch
            patch = image[start_r:end_r, start_c:end_c]

            # Select the maximum value in the patch
            output[i, j] = np.max(patch)

    return output

In [ ]:
#@title Illustrates the Convolution$\rightarrow$ReLU$\rightarrow$Max-pooling operations
# Pick an image
img_path = 'https://siamesekittykat.com/wp-content/uploads/2025/07/Siamese-Cat.png'
# img_path = 'https://c02.purpledshub.com/uploads/sites/39/2019/03/1375792762282-nt9ibl5k3vvu-5410d3b.jpg'
# img_path = 'https://muyinteresante.okdiario.com/wp-content/uploads/sites/5/2023/01/22/63cd065d4490e.jpeg'
# img_path = 'https://www.slashgear.com/img/gallery/heres-why-fords-first-production-car-was-called-the-model-t/intro-1705710312.jpg'
# img_path = 'https://cdn.britannica.com/76/116576-050-1B35C79A/Man-snowmobile-Bighorn-Mountains-Montana.jpg'
# img_path = 'https://images.squarespace-cdn.com/content/v1/657b302ad0d11e71b22b40c3/1705335532774-DAY9SFG66ATMAAZGOSVL/preview_13.normal.jpg'
# img_path = 'https://images.puppyfinder.com/Breed/1/c/b/1cb6edfcc5f4c7b9_802_image1_medium.jpg'
# img_path = 'https://www.shutterstock.com/image-photo/closeup-freshly-picked-cabbage-farmers-600nw-2671380677.jpg'
# img_path = 'https://images.stockcake.com/public/0/e/5/0e593b95-298d-454e-87e8-d1ee2f7eebf0_large/joyful-comic-reading-stockcake.jpg'
# img_path = 'https://www.shutterstock.com/image-photo/cropped-young-male-doctor-man-600nw-2717661525.jpg'
# img_path = 'https://cdn.britannica.com/80/162480-004-9DFF6461/People-beach-volleyball.jpg'
# img_path = 'https://keto-mojo.com/wp-content/uploads/2021/09/Keto-Guacamole.jpg'

# Get an imagen from an URL
IMG_SHAPE = (150, 150)
img0 = get_img_array(img_path, size=IMG_SHAPE)

# Select a filter
filter = "Laplacian" #@param ["Average", "Gaussian", "Sobel", "Laplacian", "Sharpen"]
match filter:
    case "Average":
        kernel = np.ones((3,3)) / 9 # Average
    case "Gaussian":
        kernel = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]]) / 16 # Gaussian
    case "Sobel":
        kernel = np.array([[ 1,  0, -1],[ 2,  0, -2],[ 1,  0, -1]]) # Sobel
    case "Laplacian":
        kernel = np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]]) # Laplacian
    case "Sharpen":
        kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]) # Sharpen

# Simulates the process of a convolutional layer
img1 = convolution_2d(img0, kernel)
img2 = np.maximum(img1,0) # ReLU only outputs positive values (or zero)
img3 = max_pooling_2d(img2)

# Display results
plt.figure(figsize=(12, 8))
plt.subplot(1, 4, 1)
plt.imshow(img0, cmap='gray')
plt.title('Input image')

plt.subplot(1, 4, 2)
plt.imshow(img1, cmap='gray')
plt.title('Convolution')

plt.subplot(1, 4, 3)
plt.imshow(img2, cmap='gray')
plt.title('ReLU activation')

plt.subplot(1, 4, 4)
plt.imshow(img3, cmap='gray')
plt.title('Max-pooling')

plt.tight_layout()
plt.show()